In [1]:

import pandas as pd
import tkinter as tk
from tkinter import filedialog
from pathlib import Path
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

# ============================================================
# 1. DEFINIR A COLUNA DE RESPOSTAS
# ============================================================

COLUNA = (
    "B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você "
    "sentiu vontade de se machucar de propósito?"
)

# ============================================================
# 2. SELECIONAR O ARQUIVO PELO FINDER
# ============================================================

root = tk.Tk()
root.withdraw()

arquivo = filedialog.askopenfilename(
    title="Selecione a base de dados PeNSE",
    filetypes=[
        ("Arquivos Parquet", "*.parquet")
    ]
)

root.destroy()

if not arquivo:
    raise SystemExit("Nenhum arquivo foi selecionado.")

caminho = Path(arquivo)

# ============================================================
# 3. ABRIR O ARQUIVO PARQUET
# ============================================================

parquet = pq.ParquetFile(caminho)

if COLUNA not in parquet.schema_arrow.names:
    candidatas = [
        c for c in parquet.schema_arrow.names
        if "B12009" in c
    ]

    print("Colunas encontradas com B12009:", candidatas)
    raise ValueError(
        "A coluna B12009 não foi encontrada com o nome esperado."
    )

# ============================================================
# 4. DEFINIR PASTA, DATA E HORA DOS ARQUIVOS DE SAÍDA
# ============================================================

# Pasta de trabalho atual do Jupyter
pasta_saida = Path.cwd()

# Data: dia, mês abreviado em inglês
# Hora: formato de 24 horas
agora = datetime.now()
data_hora = agora.strftime("%d_%b_%H_%M_%S").lower()

saida_sim = pasta_saida / (
    f"02_PENSE2024_IDADE_13_A_17_SIM_{data_hora}.parquet"
)

saida_nao = pasta_saida / (
    f"02_PENSE2024_IDADE_13_A_17_NAO_{data_hora}.parquet"
)

# ============================================================
# 5. SEPARAR AS RESPOSTAS EM BLOCOS
# ============================================================

escritor_sim = None
escritor_nao = None

total_sim = 0
total_nao = 0
total_outros = 0

try:
    for bloco in parquet.iter_batches(batch_size=20_000):

        # Converter o bloco para DataFrame
        df_bloco = bloco.to_pandas()

        # Normalizar a representação das respostas
        respostas = (
            df_bloco[COLUNA]
            .astype("string")
            .str.strip()
        )

        # Código 3 = SIM
        # Código 1 = NÃO
        mascara_sim = respostas.isin(["3", "3.0"])
        mascara_nao = respostas.isin(["1", "1.0"])

        # Criar tabela Arrow com todas as colunas originais
        tabela = pa.Table.from_batches([bloco])

        # Inicializar os escritores no primeiro bloco
        if escritor_sim is None:
            escritor_sim = pq.ParquetWriter(
                str(saida_sim), tabela.schema
            )

            escritor_nao = pq.ParquetWriter(
                str(saida_nao), tabela.schema
            )

        # Gravar respostas SIM
        if mascara_sim.any():
            tabela_sim = tabela.filter(
                pa.array(mascara_sim.tolist())
            )
            escritor_sim.write_table(tabela_sim)
            total_sim += int(mascara_sim.sum())

        # Gravar respostas NÃO
        if mascara_nao.any():
            tabela_nao = tabela.filter(
                pa.array(mascara_nao.tolist())
            )
            escritor_nao.write_table(tabela_nao)
            total_nao += int(mascara_nao.sum())

        # Contabilizar outras respostas e ausências
        total_outros += int(
            (~mascara_sim & ~mascara_nao).sum()
        )

finally:
    if escritor_sim is not None:
        escritor_sim.close()

    if escritor_nao is not None:
        escritor_nao.close()

# ============================================================
# 6. MOSTRAR O RESUMO FINAL
# ============================================================

print("=" * 55)
print("SEPARAÇÃO CONCLUÍDA")
print("=" * 55)

print(f"Arquivo original: {caminho}")
print(f"Total de registros: {parquet.metadata.num_rows:,}")
print(f"Respondeu SIM (3): {total_sim:,}")
print(f"Respondeu NÃO (1): {total_nao:,}")
print(f"Outras respostas/ausências: {total_outros:,}")

print("\nArquivos gerados:")

print(f"\nSIM:\n{saida_sim}")
print(f"\nNÃO:\n{saida_nao}")

print("\nPasta de saída:")
print(pasta_saida)

SEPARAÇÃO CONCLUÍDA
Arquivo original: /Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/02_Tratamento_dados_database_PENSE_2024/02_Scripts_tratamento_dados_PENSE_2024/07_PENSE2024_IDADE_13_A_17.parquet
Total de registros: 118,041
Respondeu SIM (3): 38,561
Respondeu NÃO (1): 78,789
Outras respostas/ausências: 691

Arquivos gerados:

SIM:
/Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/03_Divisao_base_dados_duas_intencao_sim_nao/02_PENSE2024_IDADE_13_A_17_SIM_18_oct_15_26_54.parquet

NÃO:
/Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/03_Divisao_base_dados_duas_intencao_sim_nao/02_PENSE2024_IDADE_13_A_17_NAO_18_oct_15_26_54.parquet

Pasta de saída:
/Users/claudiojacomassi/Documents/GitHub/Suicidio_adolescentes_Brasil/03_Divisao_base_dados_duas_intencao_sim_nao


In [1]:

import pandas as pd
import tkinter as tk
from tkinter import filedialog
from pathlib import Path
from IPython.display import display

# ============================================================
# 1. SELECIONAR OS ARQUIVOS PELO FINDER
# ============================================================

root = tk.Tk()
root.withdraw()

arquivos = filedialog.askopenfilenames(
    title="Selecione os arquivos Parquet para comparação",
    filetypes=[("Arquivos Parquet", "*.parquet")]
)

root.destroy()

if not arquivos:
    raise SystemExit("Nenhum arquivo selecionado.")

# ============================================================
# 2. CALCULAR AS ESTATÍSTICAS
# ============================================================

estatisticas = {}

for arquivo in arquivos:

    caminho = Path(arquivo)

    # Ler o arquivo
    df = pd.read_parquet(caminho)

    # Selecionar apenas as colunas numéricas
    df_numerico = df.select_dtypes(include="number")

    # Calcular count, mean e std
    resumo = df_numerico.describe().T[
        ["count", "mean", "std"]
    ]

    # Guardar o resultado usando o nome do arquivo
    estatisticas[caminho.name] = resumo

    # Liberar memória
    del df, df_numerico

# ============================================================
# 3. COMBINAR AS TABELAS LADO A LADO
# ============================================================

comparacao = pd.concat(
    estatisticas,
    axis=1
)

# ============================================================
# 4. EXIBIR A TABELA FINAL
# ============================================================

display(comparacao.round(3))


02_PENSE2024_IDADE_13_A_17_NAO_18_oct_15_26_54.parquet  \
                                                                                                    count   
REGIAO - código de regiões                                                                    78789.0       
UF - código UF                                                                                78789.0       
CAPITAL - indica se é município da capital                                                    78789.0       
B01001A - Qual é o seu sexo?                                                                  78789.0       
IDADE_AGREG - IDADE DO ESTUDANTE, CATEGORIAS                                                  78789.0       
B01002 - Qual é a sua cor ou raça?                                                            78789.0       
B01006 - Você mora com sua mãe?                                                               78789.0       
B01007 - Você mora com seu pai?                                                               78789.0       
B01015B - Na sua casa tem computador ou notebook?                                             78789.0       
B01016 - Você tem acesso à internet em sua casa?                                              78789.0       
B03006B - NOS ÚLTIMOS 7 DIAS, sem contar as aul...                                            78789.0       
B03010C - Quantas horas por dia você costuma fi...                                            78789.0       
B05002A - Alguma vez na vida você tomou um copo...                                            78789.0       
B07002 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B07004 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B07005 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B07006 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B07007B - NOS ÚLTIMOS 30 DIAS, quantas vezes al...                                            78789.0       
B07011A - NOS ÚLTIMOS 30 DIAS, quantas vezes al...                                            78789.0       
B07012A - NOS ÚLTIMOS 30 DIAS, quantas vezes al...                                            78789.0       
B07013 - NOS ÚLTIMOS 30 DIAS, você se sentiu am...                                            78789.0       
B07009 - NOS ÚLTIMOS 30 DIAS, você esculachou, ...                                            78789.0       
B07016 - NOS ÚLTIMOS 30 DIAS, você ameaçou, ofe...                                            78789.0       
B12003 - Quantos(as) amigos(as) próximos você tem?                                            78789.0       
B12004 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B12005 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B12006 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B12007 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B12008 - NOS ÚLTIMOS 30 DIAS, com que frequênci...                                            78789.0       
B12009 - NOS ÚLTIMOS 12 MESES, alguma vez você ...                                            78789.0       
B09001 - NOS ÚLTIMOS 30 DIAS, em quantos dias v...                                            78789.0       
B09002 - NOS ÚLTIMOS 30 DIAS, em quantos dias v...                                            78789.0       
B09003A - NOS ÚLTIMOS 12 MESES, quantas vezes v...                                            78789.0       
B09010A1 - NOS ÚLTIMOS 12 MESES, quantas vezes ...                                            78789.0       
B09016A1 - Alguma vez na vida alguém o(a) tocou...                                            78789.0       
B09016A2 - Alguma 